# Adding New Latent Vector Data

Zora Zorkic | December 2025

# What LVs Do we Currently Have?

---

We currently have valid data for : ['2009-01-01T12:00:00', '2025-11-17T00:00:00'] .

In [1]:
import kafou_arraylake as arraylake
import xarray as xr
import zarr
import zarr.abc.store

source_repo = "kafou/aurora-era5-t1-latent-vectors"
source_branch = "main"

client = arraylake.Client()

repo = client.get_repo(source_repo)
session = repo.readonly_session(source_branch)

ds = xr.open_zarr(
    session.store, zarr_format=3, consolidated=False, chunks=None
)

print(ds)

ds.lv.sel(time="2025-02-17T00:00:00").values

<xarray.Dataset> Size: 133TB
Dimensions:  (time: 125467, spatial_location: 259200, feature: 1024)
Coordinates:
  * time     (time) datetime64[ns] 1MB 1940-01-01T12:00:00 ... 2025-11-17
Dimensions without coordinates: spatial_location, feature
Data variables:
    lv       (time, spatial_location, feature) float32 133TB ...


array([[-0.14215225,  1.0435562 , -0.12598583, ..., -0.11196452,
         0.23670585,  0.4583925 ],
       [-0.07829029,  0.85553706, -0.09666906, ..., -0.00931154,
         0.25690648,  0.50040585],
       [-0.16430019,  1.2712905 ,  0.02004775, ..., -0.09892476,
         0.10847221,  0.44431874],
       ...,
       [-0.36898112,  1.4300619 , -0.7770787 , ...,  0.3769868 ,
         0.5689063 ,  0.16636291],
       [-0.40453476,  2.072168  , -0.61933887, ...,  0.37770846,
         0.42500842,  0.3190994 ],
       [-0.41205055,  1.052494  , -0.4681805 , ...,  0.30136892,
         0.6317799 ,  0.00514189]], shape=(259200, 1024), dtype=float32)

# What CHANNELIZED ERA5 timestamps do we have for Auora LV calculations ?

---

We currently have valid channelized data for: ['1940-01-01T00:00:00', '2025-11-17T18:00:00'].

In [3]:
import kafou_arraylake as arraylake
import zarr
import numpy as np

repo_name = "kafou/aurora-era5-samples"
branch = "extend-2025"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)
ds = xr.open_zarr(
    ro.store,
    group="samples",
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)

ds.sample_data.sel(time="2025-03-31T00:00:00").values

<xarray.Dataset> Size: 36TB
Dimensions:       (time: 125472, channel: 69, latitude: 721, longitude: 1440,
                   atmos_levels: 13)
Coordinates:
  * atmos_levels  (atmos_levels) int64 104B 50 100 150 200 ... 700 850 925 1000
  * time          (time) datetime64[ns] 1MB 1940-01-01 ... 2025-11-17T18:00:00
  * longitude     (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
  * latitude      (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * channel       (channel) int32 276B 0 1 2 3 4 5 6 7 ... 62 63 64 65 66 67 68
Data variables:
    sample_data   (time, channel, latitude, longitude) float32 36TB ...
Attributes:
    var_locs:  {'sfc': {'2t': [0, 1], 'msl': [1, 1], '10u': [2, 1], '10v': [3...


array([[[ 2.48493103e+02,  2.48493103e+02,  2.48493103e+02, ...,
          2.48493103e+02,  2.48493103e+02,  2.48493103e+02],
        [ 2.48153259e+02,  2.48155212e+02,  2.48157166e+02, ...,
          2.48147400e+02,  2.48149353e+02,  2.48151306e+02],
        [ 2.47989197e+02,  2.47993103e+02,  2.47997009e+02, ...,
          2.47979431e+02,  2.47983337e+02,  2.47987244e+02],
        ...,
        [ 2.21788025e+02,  2.21791931e+02,  2.21795837e+02, ...,
          2.21780212e+02,  2.21782166e+02,  2.21786072e+02],
        [ 2.22053650e+02,  2.22055603e+02,  2.22057556e+02, ...,
          2.22049744e+02,  2.22051697e+02,  2.22051697e+02],
        [ 2.22196228e+02,  2.22196228e+02,  2.22196228e+02, ...,
          2.22196228e+02,  2.22196228e+02,  2.22196228e+02]],

       [[ 1.01556875e+05,  1.01556875e+05,  1.01556875e+05, ...,
          1.01556875e+05,  1.01556875e+05,  1.01556875e+05],
        [ 1.01607875e+05,  1.01607625e+05,  1.01607625e+05, ...,
          1.01607875e+05,  1.01607875e

ds.sample_data.sel(time="2025-03-25T00:00:00").values

In [6]:
import numpy as np

times_2025 = ds.time.sel(time=slice("2025-03-25", "2025-12-31"))

nan_times = []

for t in times_2025.values:
    arr = ds.sample_data.sel(time=t).values  # single timestep only
    if np.isnan(arr).any():
        nan_times.append(t)
        print(f"NaNs found at {t}")

nan_times


[]

# What New Reanalysis ERA5 Variables do we have ?

---

We currently have RAW ERA5 data from: ['1940-01-01T00:00:00', '2025-12-09T18:00:00'].

In [27]:
import kafou_arraylake as arraylake
import xarray as xr
import zarr
import numpy as np

source_repo = "rwe/era5-0p25-6h-nonprod-ohio"
source_branch = "main"

client = arraylake.Client()
repo = client.get_repo(source_repo)
session = repo.readonly_session(source_branch)

sfc_ds = xr.open_zarr(session.store, group="surface")  # no zarr_format override

# get the time range of the dataset in np datetiem objects
raw_min = np.datetime_as_string(sfc_ds.time.min().values, unit="s")
raw_max = np.datetime_as_string(sfc_ds.time.max().values, unit="s")

reanalysis_time_range = [str(raw_min), str(raw_max)]
print(reanalysis_time_range)

sfc_ds


/tmp/ipykernel_2312957/217266576.py:13: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  sfc_ds = xr.open_zarr(session.store, group="surface")  # no zarr_format override


['1940-01-01T00:00:00', '2025-12-10T18:00:00']


<xarray.Dataset> Size: 10TB
Dimensions:            (time: 125564, latitude: 721, longitude: 1440, level: 13)
Coordinates:
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * level              (level) float64 104B 50.0 100.0 150.0 ... 925.0 1e+03
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
  * time               (time) datetime64[ns] 1MB 1940-01-01 ... 2025-12-10T18...
Data variables: (12/20)
    STL3               (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    CI                 (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    SD                 (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    BLH                (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    MSL                (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    STL1               (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    ...                 ...
    VAR_10U            (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    VAR_100U           (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    VAR_2T             (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    VAR_10V            (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    SKT                (time, latitude, longitude) float32 521GB dask.array<chunksize=(4, 256, 256), meta=np.ndarray>
    quantization_info  object 8B ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.9.2
    CONVERSION_PLATFORM:  Linux crhtc81 5.14.21-150400.24.46-default #1 SMP P...
    CONVERSION_DATE:      Sat 31 May 2025 03:38:52 PM MDT
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Sat May 31 15:39:00 2025: ncks -4 -L 1 --baa=0 --pp...
    NCO:                  netCDF Operators version 5.3.1 (Homepage = http://n...

# Now What?

---

1. We could add channelized data from  2025-11-17TT00:00:00 to 2025-12-09T18:00:00. 
2. Then we need to generate LVs from 2025-11-17TT00:00:00 to 2025-11-17T18:00:00. 


# Get ERA5 Sample data ready for inference 

## Create a New Branch to hold channelized data

In [59]:
# import kafou_arraylake as arraylake

# new_branch_name = "extend-2025"
# client = arraylake.Client()
# repo = client.get_repo("kafou/aurora-era5-samples")
# # repo.delete_branch(new_branch_name)

# branches = repo.list_branches()
# print("Branches:", branches)
# # 1. Lookup the existing branch we want to fork from
# main_info = repo.lookup_branch("main")
# print("Main branch info:", main_info)
# # 2. Extract snapshot ID
# snapshot_id = repo.lookup_snapshot('HVB13KF5MQWGF3MJGZRG')
# print("Snapshot ID:", snapshot_id)
# print("Snapshot info:", snapshot_id.id)
# # 3. Create the new branch
# repo.create_branch(new_branch_name, main_info)
# print("New branch created!")
# branches = repo.list_branches()
# print("Branches:", branches)


Branches: {'test-one-day', 'main'}
Main branch info: HVB13KF5MQWGF3MJGZRG
Snapshot ID: SnapshotInfo(id="HVB13KF5MQWGF3MJGZRG", parent_id=KVQWZ96N948EFSCAKVHG, written_at=datetime.datetime(2025,7,8,15,26,8,315563, tzinfo=datetime.timezone.utc), message="Commit...")
Snapshot info: HVB13KF5MQWGF3MJGZRG
New branch created!
Branches: {'test-one-day', 'main', 'extend-2025'}


In [ ]:
# RUN resume_era5.py

In [6]:
import kafou_arraylake as arraylake
import xarray as xr

source_repo = "kafou/aurora-era5-samples"

client = arraylake.Client()

repo = client.get_repo(source_repo)


session = repo.readonly_session("extend-2025")
sample_ds = xr.open_zarr(session.store, group="samples", zarr_format=3, consolidated=False, chunks=None)

sample_ds

<xarray.Dataset> Size: 36TB
Dimensions:       (longitude: 1440, atmos_levels: 13, time: 125472,
                   latitude: 721, channel: 69)
Coordinates:
  * longitude     (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
  * atmos_levels  (atmos_levels) int64 104B 50 100 150 200 ... 700 850 925 1000
  * time          (time) datetime64[ns] 1MB 1940-01-01 ... 2025-11-17T18:00:00
  * latitude      (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
Dimensions without coordinates: channel
Data variables:
    sample_data   (time, channel, latitude, longitude) float32 36TB ...
Attributes:
    var_locs:  {'sfc': {'2t': [0, 1], 'msl': [1, 1], '10u': [2, 1], '10v': [3...

In [26]:
import kafou_arraylake as arraylake
import xarray as xr

source_repo = "kafou/aurora-ecmwf-samples"

client = arraylake.Client()

repo = client.get_repo(source_repo)


session = repo.readonly_session("main")
sample_ds = xr.open_zarr(session.store, group="samples", zarr_format=3, consolidated=False, chunks=None)

ds.lv.sel(time="2025-11-17").values

array([[[-0.2186014 , -0.20955092, -0.0111398 , ..., -0.16236997,
          0.0136869 , -0.16117024],
        [-0.08187763, -0.33080515, -0.03815991, ..., -0.07497343,
          0.06727084, -0.12393618],
        [-0.20693322,  0.04131094,  0.18353823, ..., -0.17006508,
         -0.06603856, -0.17345995],
        ...,
        [-0.21519175,  0.65011346, -0.59607154, ..., -0.07831296,
          0.06857844,  0.45704496],
        [-0.12327552,  1.3237457 , -0.5608453 , ..., -0.08958905,
         -0.0762138 ,  0.6009389 ],
        [-0.2057327 ,  0.31109777, -0.28651822, ..., -0.14983414,
          0.11684403,  0.27947748]]],
      shape=(1, 259200, 1024), dtype=float32)

# Get Updated LVs

## Create New LV branch to hold extended LVs

In [57]:
# add valid_time_range attribute to latent vectors dataset
repo = client.get_repo("kafou/aurora-era5-t1-latent-vectors")

session = repo.readonly_session("december-2025")
ds = xr.open_zarr(
    session.store, zarr_format=3, consolidated=False, chunks=None
)

ds

<xarray.Dataset> Size: 133TB
Dimensions:  (time: 125467, spatial_location: 259200, feature: 1024)
Coordinates:
  * time     (time) datetime64[ns] 1MB 1940-01-01T12:00:00 ... 2025-11-17
Dimensions without coordinates: spatial_location, feature
Data variables:
    lv       (time, spatial_location, feature) float32 133TB ...
Attributes:
    valid_time_range:  ['2009-01-01T12:00:00', '2025-11-17T00:00:00']